Objetivo

En esta notebook se utilizará el modelo Random Forest seleccionado en el notebook 05_validacion para realizar predicciones sobre un nuevo conjunto de datos sin etiquetar.

Para ello, se aplicará el mismo preprocesamiento utilizado durante el entrenamiento y se generará una nueva columna llamada smoking_prediction que indicará si cada persona es fumadora (1) o no fumadora (0).

In [1]:
import pandas as pd
import joblib

In [2]:
# Cargar el modelo entrenado
modelo_final = joblib.load(
    "../models/random_forest_final.joblib"
)

In [3]:
# Reemplazar por el nombre real del segundo dataset
df_nuevo = pd.read_csv(
    "../data/raw/smoking_prediction_entrega.csv"
)

print("Dimensiones:", df_nuevo.shape)

display(df_nuevo.head())

print("\nColumnas:")
print(df_nuevo.columns.tolist())

Dimensiones: (5692, 26)


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.42,0.05,0.00,0.04,0.04,...,3.04,0.63,0.04,0.01,0.75,0.71,0.71,Y,0,Y
1,27364,M,30,180,80,3.46,0.04,0.01,0.04,0.04,...,4.21,0.59,0.04,0.04,0.79,1.13,1.33,Y,0,N
2,27368,M,55,165,60,3.42,0.00,0.01,0.04,0.04,...,2.04,0.63,0.04,0.01,1.08,1.29,2.00,Y,1,Y
3,27378,M,20,175,75,3.63,0.05,0.05,0.04,0.04,...,3.71,0.63,0.04,0.04,0.83,0.58,0.46,Y,0,N
4,27381,M,25,165,80,3.79,0.04,0.04,0.04,0.04,...,6.63,0.67,0.04,0.04,1.25,1.63,1.96,Y,1,Y



Columnas:
['ID', 'gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'oral', 'dental caries', 'tartar']


In [4]:
# Guardamos el ID para identificar cada predicción
ids = df_nuevo["ID"].copy()

# Eliminamos las columnas que no utilizamos para entrenar
X_nuevo = df_nuevo.drop(columns=["ID", "oral"])

print("Dimensiones:", X_nuevo.shape)

Dimensiones: (5692, 24)


In [5]:
# Carga de los transformadores utilizados en el preprocesamiento
encoder = joblib.load("../models/encoder.joblib")
scaler = joblib.load("../models/scaler.joblib")

#### OneHotEncoder

In [6]:
columnas_categoricas = ["gender", "tartar"]

# Transformamos las variables categóricas
datos_encoded = encoder.transform(
    X_nuevo[columnas_categoricas]
)

# Recuperamos los nombres de las columnas generadas
nombres_columnas = encoder.get_feature_names_out(
    columnas_categoricas
)

# Creamos un DataFrame con las columnas codificadas
df_encoded = pd.DataFrame(
    datos_encoded,
    columns=nombres_columnas,
    index=X_nuevo.index
)

# Eliminamos las variables originales y agregamos las codificadas
X_nuevo = pd.concat(
    [
        X_nuevo.drop(columns=columnas_categoricas),
        df_encoded
    ],
    axis=1
)

print("Dimensiones:", X_nuevo.shape)

Dimensiones: (5692, 26)


#### MinMaxScaler

In [7]:
columnas_numericas = scaler.feature_names_in_

X_nuevo[columnas_numericas] = scaler.transform(
    X_nuevo[columnas_numericas]
)

In [8]:
X_train = joblib.load("../data/processed/X_train.joblib")

# Ordenar las columnas igual que en entrenamiento
X_nuevo = X_nuevo[X_train.columns]

print("Dimensiones finales:", X_nuevo.shape)
print("¿Mismas columnas?", X_nuevo.columns.equals(X_train.columns))
print("¿Valores nulos?", X_nuevo.isnull().sum().sum())

Dimensiones finales: (5692, 26)
¿Mismas columnas? True
¿Valores nulos? 0


#### Predicciones

In [9]:
predicciones = modelo_final.predict(X_nuevo)

df_resultado = pd.DataFrame({
    "ID": ids,
    "smoking_prediction": predicciones
})

display(df_resultado.head())
print("Cantidad de predicciones:", len(predicciones))
print("Valores encontrados:", pd.Series(predicciones).unique())

,ID,smoking_prediction
0,27358,0
1,27364,1
2,27368,1
3,27378,0
4,27381,1


Cantidad de predicciones: 5692
Valores encontrados: [0 1]


In [10]:
df_resultado.to_csv(
    "../data/processed/predicciones.csv",
    index=False
)

print("Cantidad de registros:", len(df_resultado))
print("Cantidad de IDs únicos:", df_resultado["ID"].nunique())
print("Valores nulos:", df_resultado.isnull().sum().sum())
print("Predicciones:")
print(df_resultado["smoking_prediction"].value_counts())

print("Predicciones guardadas correctamente")

Cantidad de registros: 5692
Cantidad de IDs únicos: 5692
Valores nulos: 0
Predicciones:
smoking_prediction
0    3328
1    2364
Name: count, dtype: int64
Predicciones guardadas correctamente


Se utilizó el modelo Random Forest controlado para predecir la variable smoking sobre 5.692 registros nuevos.

Se aplicaron las mismas transformaciones utilizadas durante el entrenamiento y se verificó que las 26 columnas coincidieran con las esperadas por el modelo.

El modelo clasificó a 3.328 personas como no fumadoras (58,47 %) y a 2.364 como fumadoras (41,53 %).

Finalmente, se generó el archivo predicciones.csv con las columnas ID y smoking_prediction, sin registros duplicados ni valores nulos.